# Qwen2.5-7B Laravel Coder — Inference

Loads merged weights if available, otherwise base model + LoRA adapter.

In [ ]:
!pip install -q transformers accelerate peft

In [ ]:
import os
import tarfile
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
HF_MODEL = "bhavin-gajjar/qwen-laravel-coder"
INPUT = Path("/kaggle/input")

def find_weights_root() -> Path | None:
    for p in [
        INPUT / "qwen-laravel-coder-weights",
        INPUT / "datasets" / "bhavingajjar22" / "qwen-laravel-coder-weights",
        *INPUT.rglob("qwen-laravel-coder-weights"),
    ]:
        if p.is_dir():
            for tar in p.glob("*.tar"):
                with tarfile.open(tar) as tf:
                    tf.extractall(p)
            return p
    return None

weights = find_weights_root()
merged = None
adapter = None
if weights:
    merged = next((x for x in weights.rglob("config.json") if (x.parent / "model.safetensors").exists()), None)
    adapter = next((x.parent for x in weights.rglob("adapter_config.json")), None)

cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16

if merged:
    MODEL = str(merged.parent)
    print(f"Merged: {MODEL}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=DTYPE, device_map="auto", trust_remote_code=True)
elif adapter:
    print(f"Adapter: {adapter}")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=DTYPE, device_map="auto", trust_remote_code=True)
    model = PeftModel.from_pretrained(base, str(adapter))
else:
    print(f"HF fallback: {HF_MODEL}")
    tokenizer = AutoTokenizer.from_pretrained(HF_MODEL, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(HF_MODEL, torch_dtype=DTYPE, device_map="auto", trust_remote_code=True)

In [ ]:
messages = [{"role": "user", "content": "How do I register middleware in Laravel 11?"}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=512)
print(tokenizer.decode(out[0], skip_special_tokens=True))